# Assignment 2 — classifier training on Colab

Reproduction notebook for Matej's classification contribution. The recorded ViT experiment used a Colab Tesla T4; this notebook provides the same command-line pipeline in cells. Recorded results are in the repository and release assets. This notebook is not an additional measured experiment.

Select **Runtime → Change runtime type → T4 GPU** before running. A complete ViT run takes about two hours on the measured T4. Colab session storage is temporary: download the results before disconnecting. Detection work and the combined team submission are tracked separately in `TEAM_PLAN.md`.


In [ ]:
from pathlib import Path
import os, subprocess, sys

REPOSITORY = 'https://github.com/matejpopovski/bmi771-assignment2.git'
REVISION = 'main'  # Set to a recorded commit to reproduce a fixed code revision.
ROOT = Path('/content/bmi771-classification')
EXPERIMENTS = ['vit']  # Other choices: 'simple', 'cnn', 'resnet18'.

if not ROOT.exists():
    subprocess.run(['git', 'clone', REPOSITORY, str(ROOT)], check=True)
os.chdir(ROOT)
subprocess.run(['git', 'checkout', REVISION], check=True)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-r', 'code/requirements.txt'], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before training.'
print('Device:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)


In [ ]:
subprocess.run([sys.executable, 'code/download_datasets.py', '--classification-only'], check=True)
(ROOT/'results').mkdir(exist_ok=True)
env = dict(os.environ, OMP_NUM_THREADS='2')
with open(ROOT/'results/cloud_tests.txt', 'w') as stream:
    subprocess.run([sys.executable, '-m', 'pytest', 'code/tests', '-q'],
                   stdout=stream, stderr=subprocess.STDOUT, env=env, check=True)
print((ROOT/'results/cloud_tests.txt').read_text())


## Train and evaluate

The runner saves resolved configurations, epoch histories, TensorBoard events and checkpoints under `logs/`, then evaluates and copies final artifacts into `results/`. Existing final results are skipped, and intermediate checkpoints can be resumed. For a fresh independent run, use a separate directory. Downloaded repository metrics alone do not count as a completed run; a final checkpoint is also required.

For live progress, open Colab's terminal and inspect `logs/cls_miniplaces_vit_run_assignment2/training.log`. Do not close or delete the runtime before saving its output.


In [ ]:
subprocess.run([sys.executable, '-u', 'code/run_experiments.py',
                '--device', 'cuda:0', '--workers', '2', '--experiments', *EXPERIMENTS],
               env=env, check=True)


In [ ]:
import hashlib, json, zipfile
from google.colab import files

with open(ROOT/'results/environment_cloud.txt', 'w') as stream:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=stream, check=True)
archive = Path('/content/classifier-cloud-results.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in EXPERIMENTS:
        folder = ROOT/'results'/name
        assert (folder/'model_final.pth.tar').is_file(), f'Missing final {name} weights'
        metrics = json.loads((folder/'metrics.json').read_text())
        assert metrics['num_images'] == 10000
        print(name, metrics)
        for path in folder.rglob('*'):
            if path.is_file(): z.write(path, path.relative_to(ROOT))
    for filename in ('cloud_tests.txt', 'environment_cloud.txt'):
        path = ROOT/'results'/filename
        z.write(path, path.relative_to(ROOT))
print('SHA-256:', hashlib.sha256(archive.read_bytes()).hexdigest())
files.download(str(archive))


After downloading, verify the printed checksum and extract the archive into the assignment directory. Run `code/recheck_classification.py` to independently evaluate the saved weights, then `code/verify_classification.py` to audit all four classifiers. Disconnect the Colab runtime when finished. The classifier component must still be integrated with the teammate's verified detector work before Canvas submission.
